# Oracle (V5) — Test rapide de décalage climatique (OOD directionnel)

**Contexte.** Le jeu de données du projet (Zenodo 10889046, vérifié) ne contient
**aucun scénario futur (SSP)** : le vrai test OOD climatique du rapport
d'évaluation (§5.3) est hors de portée avec ces données. Ce notebook exécute
les deux meilleurs proxys **directionnels** possibles avec l'historique
1960--2014, sachant que le modèle n'a vu que 1980--2009 à l'entraînement :

| Tranche | Période | Nature |
|---|---|---|
| `id_2012_2013` | 2012--2013 | référence en distribution (recalculée ici même, mêmes N/K/pas) |
| `cold_1960_1965` | 1960--1965 | **décalage climatique réel** : 20 ans avant le train, climat plus froid, jamais vu |
| `warm_tail_2012_2014` | jours les plus chauds de 2012--2014 (z-t850 domaine, top ~7 %) | queue chaude **hors train et hors validation** : direction du réchauffement |

**Question mesurée** : la *pente de dégradation* — quand la distribution
climatique se décale, le CRPS d'Oracle se dégrade-t-il moins vite que celui de
la baseline Noncausal ?

**Garde-fous intégrés** : Oracle = checkpoint V5 baseline explicite (aucun
fallback fine-tuné) ; normalisation figée sur les statistiques d'entraînement
(aucun réajustement sur les tranches) ; assert sur l'activation réelle du
découpage temporel (sinon le pipeline ignorerait le split en silence) ;
vérification climatique finale (z-t850 : cold < id < warm) ; traçabilité des
checkpoints dans le JSON.

**Lecture honnête** : signaux directionnels (froid réel ~0,3--0,6 K, queue
chaude intra-historique), pas le scénario SSP5-8.5.

**Coût estimé** : ~30--35 min sur A100 ; ~1h--1h30 sur L4 (réduire `N_DAYS` à
90 et `K_ENS` à 6 pour rester sous l'heure).

**Sorties** : `RESULTS_DIR/ood_climate_shift.json` + `.png`.

In [ ]:
# >>> COLAB_BOOTSTRAP
# Bootstrap Colab optimisé — premier run ~3 min, re-runs ~30 s.
# Stratégie :
#   • Code sur SSD local (/content/) — git clone 5-10× plus rapide que vers Drive.
#   • Drive UNIQUEMENT pour les checkpoints (cf. cellule helpers plus bas).
#   • Pas de ``pip install -r requirements.txt`` brut (déclenche la compilation
#     CUDA de torch-scatter/torch-sparse → 20-30 min). À la place : install
#     pinned des seules deps non pré-installées par Colab.
#   • Wheels PyG pré-construits via le bon index (sinon torch-scatter/torch-sparse
#     compilent depuis les sources — interdit ici).
#
# Hors Colab : no-op.
import os, sys, subprocess, time, shlex
from pathlib import Path

# ── Configuration utilisateur ──────────────────────────────────────────
GIT_URL: str | None = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH: str = "two-stage-causal"  # hyperplan v2.0  # Phase 1 EDM rewrite (Karras 2022)
LOCAL_PROJECT = "/content/climate_data"  # SSD — toujours rapide
GIT_PULL_ON_RESUME = True
SKIP_PIP_IF_IMPORTABLE = True  # si ``import st_cdgm`` réussit déjà → skip pip

_IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()


def _run(cmd: str, *, check: bool = True, timeout: int | None = None) -> int:
    """Exécute une commande shell avec timing visible."""
    print(f"$ {cmd}")
    t0 = time.time()
    rc = subprocess.call(shlex.split(cmd), timeout=timeout)
    dt = time.time() - t0
    print(f"  ↳ rc={rc}  ({dt:.1f}s)")
    if check and rc != 0:
        raise RuntimeError(f"Commande échouée : {cmd!r} (rc={rc})")
    return rc


if _IS_COLAB:
    _T0 = time.time()
    print("🛰️  Colab détecté — bootstrap en cours…\n")

    # 1) Monter Drive (idempotent — pour la cellule de persistance plus loin)
    from google.colab import drive  # type: ignore[import-not-found]
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    else:
        print("   /content/drive déjà monté.")

    # 2) Clone vers SSD (PAS vers Drive — FUSE est lent pour des milliers de petits fichiers)
    project_path = Path(LOCAL_PROJECT)
    if not (project_path / ".git").exists():
        if GIT_URL is None:
            raise RuntimeError(
                "GIT_URL=None et projet absent du SSD. Renseignez GIT_URL ci-dessus, "
                "ou pré-uploadez le projet à " + LOCAL_PROJECT + " avant ce run."
            )
        project_path.parent.mkdir(parents=True, exist_ok=True)
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}")
    elif GIT_PULL_ON_RESUME:
        try:
            _run(f"git -C {LOCAL_PROJECT} pull --ff-only", timeout=60, check=False)
        except Exception as e:
            print(f"   ⚠️  git pull a levé : {e}")

    # 3) cd dans la racine — config/, src/, etc. en chemins relatifs
    os.chdir(project_path)
    print(f"   chdir → {os.getcwd()}\n")

    # 4) Test d'import — si st_cdgm marche déjà, on saute pip (énorme gain au re-run)
    src_path = str(project_path / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    _need_pip = True
    if SKIP_PIP_IF_IMPORTABLE:
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            from diffusers import UNet2DConditionModel  # noqa: F401
            import torch_geometric  # noqa: F401
            _need_pip = False
            print("✓ Imports critiques OK — pip install sauté.")
        except ImportError as _imp_err:
            print(f"   import st_cdgm a échoué ({_imp_err}) — pip install requis.")

    if _need_pip:
        # 5) Versions PyTorch / CUDA déjà installées par Colab
        import torch
        TORCH_VER = torch.__version__.split("+")[0]  # ex. "2.5.1"
        TORCH_TAG = f"torch-{TORCH_VER}"             # ex. "torch-2.5.1"
        CUDA_TAG = "cu" + (torch.version.cuda or "121").replace(".", "") if torch.cuda.is_available() else "cpu"
        print(f"   torch={TORCH_VER}, cuda={CUDA_TAG}\n")

        # 6) Install des deps NON pré-installées par Colab
        # (numpy/pandas/scipy/sklearn/matplotlib/torch/torchvision/torchaudio/
        #  zarr/dask/xarray/h5netcdf/cartopy/tqdm/requests/ipykernel sont déjà là)
        EXTRA_DEPS = [
            "omegaconf==2.3.0",
            "hydra-core==1.3.2",
            "diffusers==0.36.0",
            "transformers==4.57.6",
            "accelerate==1.12.0",
            "huggingface-hub==0.36.0",
            "safetensors==0.7.0",
            "xbatcher",
            "webdataset",
            "cftime",
            "h5netcdf",
            "numcodecs",
            "torch-geometric",  # v2.3+ ne nécessite plus torch-scatter/torch-sparse
            "xformers",  # OOM fix R2 : memory-efficient attention sur sm_75 (T4)
        ]
        deps_str = " ".join(shlex.quote(p) for p in EXTRA_DEPS)
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location {deps_str}",
            timeout=600,
        )

        # 7) torch-scatter / torch-sparse — *optionnels* avec PyG ≥ 2.3 (fallback
        # pure-PyTorch). Décommentez si vous tombez sur un module qui les exige.
        # _PYG_INDEX = f"https://data.pyg.org/whl/{TORCH_TAG}+{CUDA_TAG}.html"
        # _run(
        #     f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
        #     f"torch-scatter torch-sparse -f {_PYG_INDEX}",
        #     timeout=600, check=False,
        # )

        # 8) Editable install du package — --no-deps pour ne PAS retomber sur
        # requirements.txt (qui réinstallerait torch et compilerait torch-scatter).
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
            f"--no-deps -e {LOCAL_PROJECT}",
            timeout=120,
        )

        # 9) Re-test des imports critiques
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            print("✓ st_cdgm importable.")
        except ImportError as e:
            print(f"⚠️  st_cdgm pas encore importable depuis ce kernel : {e}")
            print("   → Probablement un cache d'import — Runtime → Restart runtime puis re-run.")

    print(f"\n✅ Bootstrap Colab terminé en {time.time() - _T0:.1f}s.")

else:
    # Hors Colab : remonte automatiquement à la racine projet.
    _here = Path.cwd()
    for _candidate in [_here, *_here.parents]:
        if (_candidate / "config" / "training_config.yaml").exists() and (_candidate / "setup.py").exists():
            if _candidate != _here:
                os.chdir(_candidate)
                print(f"📂 chdir → {os.getcwd()} (racine projet détectée)")
            break
    print("ℹ️  Hors Colab — bootstrap sauté (assume install déjà faite).")

# === V5 specific smoke test (apres bootstrap) ===
try:
    from st_cdgm.models import ConditionalSkipBlock
    print("[Oracle smoke] ConditionalSkipBlock importable - Oracle features pretes")
except ImportError as e:
    print(f"[Oracle smoke] ConditionalSkipBlock NON disponible : {e}")
    print("           Verifier que la branche two-stage-causal contient src/st_cdgm/models/skip_direct.py")

try:
    from scripts.intervention_test import INTERVENTIONS
    print(f"[Oracle smoke] Phase 8 helpers OK ({len(INTERVENTIONS)} interventions)")
except ImportError as e:
    print(f"[Oracle smoke] scripts.intervention_test NON disponible : {e}")


In [ ]:
# === CONFIGURATION DES CHEMINS — Oracle = V5 BASELINE du memoire ===
# (correction d'audit : l'ancienne cellule pointait oracle_finetuned/ si ce
#  dossier existait sur le Drive ; ici on charge explicitement le checkpoint
#  V5 baseline qui a produit les chiffres du memoire, sans aucun fallback.)
from pathlib import Path
import hashlib

V5_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_v2_corrdiff_normal")
NONCAUSAL_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_noncausal")

ORACLE_DIR = V5_DIR                 # utilise par le bootstrap autonome
CHECKPOINT_NAME = "epoch_last"      # idem

RESULTS_DIR = Path("/content/drive/MyDrive/climate_data/results/oracle_evaluation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_INFO = {}
for label, p in [("Oracle (V5 baseline)", ORACLE_DIR / f"{CHECKPOINT_NAME}.pth"),
                 ("Noncausal", NONCAUSAL_DIR / "epoch_last.pth")]:
    if not p.exists():
        raise FileNotFoundError(f"{label} : checkpoint ABSENT -> {p}")
    size_gb = p.stat().st_size / 1e9
    CKPT_INFO[label] = {"path": str(p), "size_gb": round(size_gb, 3)}
    print(f"  {label:22s}: OK ({size_gb:.2f} GB)  {p}")
print(f"\nResultats -> {RESULTS_DIR}")


In [ ]:
# ============================================================
# Bootstrap autonome COMPLET (= Cells 15+16+17+30+32+40 du training notebook)
# Telecharge automatiquement TOUS les datasets manquants depuis Zenodo
# ============================================================
import os
import sys
import time
import json
import shutil
import urllib.request
import urllib.error
import torch
import numpy as np
from pathlib import Path
from omegaconf import OmegaConf

ON_COLAB = "google.colab" in sys.modules or Path("/content").exists()

# 1. CONFIG (base + override corrdiff_normal si dispo)
_base = Path("config/training_config.yaml")
_override = Path("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.load(_base)
if _override.exists():
    CONFIG = OmegaConf.merge(CONFIG, OmegaConf.load(_override))
    print("[OK] CONFIG = base + corrdiff_normal override")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
print(f"[OK] DEVICE={DEVICE}  lr_shape={lr_shape}  hr_shape={hr_shape}")

# 2. DATA_ROOT detection (= training cell 16)
DATA_ROOT_LOCAL = Path("data/raw")
DATA_ROOT_DRIVE = Path("/content/drive/MyDrive/climate_data/data")
_DATA_ROOT_LOCAL_SSD = Path("/content/data_local")

if ON_COLAB and DATA_ROOT_DRIVE.parent.parent.exists():
    DATA_ROOT = DATA_ROOT_DRIVE
    print(f"[INFO] DATA_ROOT = Drive ({DATA_ROOT})")
else:
    DATA_ROOT = DATA_ROOT_LOCAL
    print(f"[INFO] DATA_ROOT = local ({DATA_ROOT.resolve()})")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 3. BS32 SSD copy (Drive -> /content/data_local pour I/O rapide)
_BS32_ENABLED = bool(globals().get("DATA_LOCAL_SSD", True))
if ON_COLAB and _BS32_ENABLED and DATA_ROOT == DATA_ROOT_DRIVE:
    _files_to_copy = [
        ("train/predictor_ACCESS-CM2_hist.nc",   "predictor_ACCESS-CM2_hist.nc"),
        ("train/pr_ACCESS-CM2_hist.nc",          "pr_ACCESS-CM2_hist.nc"),
        ("static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc",
         "ERA5_eval_ccam_12km.198110_NZ_Invariant.nc"),
        ("normalization_coefs/mean_1974_2011.nc", "mean_1974_2011.nc"),
        ("normalization_coefs/std_1974_2011.nc",  "std_1974_2011.nc"),
    ]
    _ssd_train = _DATA_ROOT_LOCAL_SSD / "train"
    _ssd_static = _DATA_ROOT_LOCAL_SSD / "static_predictors"
    _ssd_norm = _DATA_ROOT_LOCAL_SSD / "normalization_coefs"
    for _d in (_ssd_train, _ssd_static, _ssd_norm):
        _d.mkdir(parents=True, exist_ok=True)
    _t_total = time.time()
    _bytes_copied = 0
    for _rel, _name in _files_to_copy:
        _src = DATA_ROOT_DRIVE / _rel
        if "train/" in _rel:
            _dst = _ssd_train / _name
        elif "static_predictors/" in _rel:
            _dst = _ssd_static / _name
        else:
            _dst = _ssd_norm / _name
        if not _src.exists():
            continue
        if _dst.exists() and _dst.stat().st_size == _src.stat().st_size:
            continue
        _t0 = time.time()
        print(f"   copie {_src.name}...", flush=True)
        shutil.copy2(_src, _dst)
        _bytes_copied += _dst.stat().st_size
        print(f"   OK {_dst.name} ({_dst.stat().st_size/1e6:.0f} MB en {time.time()-_t0:.1f}s)")
    if _bytes_copied > 0:
        print(f"BS32 SSD copy: {_bytes_copied/1e9:.2f} GB en {time.time()-_t_total:.1f}s")
    DATA_ROOT = _DATA_ROOT_LOCAL_SSD
    print(f"[INFO] DATA_ROOT redirige vers SSD : {_DATA_ROOT_LOCAL_SSD}")

# 4. Resolution des paths (= training cell 16 suite)
def _relocate(p):
    if not p:
        return p
    s = str(p)
    if s.startswith("data/raw/"):
        return str(DATA_ROOT / s[len("data/raw/"):])
    return s

for _key in ("lr_path", "hr_path", "static_path"):
    if CONFIG.data.get(_key):
        CONFIG.data[_key] = _relocate(CONFIG.data[_key])

LR_PATH = str(CONFIG.data.lr_path)
HR_PATH = str(CONFIG.data.hr_path)
STATIC_PATH = str(CONFIG.data.static_path) if CONFIG.data.get("static_path") else None
MEAN_PATH = str(DATA_ROOT / "normalization_coefs" / "mean_1974_2011.nc")
STD_PATH = str(DATA_ROOT / "normalization_coefs" / "std_1974_2011.nc")

URL_ZENODO_HR = "https://zenodo.org/records/10889046/files/pr_ACCESS-CM2_hist.nc?download=1"
URL_ZENODO_LR = "https://zenodo.org/records/10889046/files/predictor_ACCESS-CM2_hist.nc?download=1"
URLS_TEST = [
    ("EC-Earth3_histupdated_compressed.nc",
     "https://zenodo.org/records/10889046/files/EC-Earth3_histupdated_compressed.nc?download=1"),
    ("EC-Earth3_historical_precip_compressed.nc",
     "https://zenodo.org/records/10889046/files/EC-Earth3_historical_precip_compressed.nc?download=1"),
    ("NorESM2-MM_histupdated_compressed.nc",
     "https://zenodo.org/records/10889046/files/NorESM2-MM_histupdated_compressed.nc?download=1"),
    ("NorESM2-MM_historical_precip_compressed.nc",
     "https://zenodo.org/records/10889046/files/NorESM2-MM_historical_precip_compressed.nc?download=1"),
]

# 5. stream_download (atomique + reprise + timeout)
def stream_download(url, dest, retries=5, chunk_size=1024*1024,
                    connect_timeout=30, read_timeout=120):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        already = part.stat().st_size if part.exists() else 0
        req = urllib.request.Request(url)
        if already > 0:
            req.add_header("Range", f"bytes={already}-")
            print(f"   reprise a {already/1e6:.1f} MB")
        try:
            with urllib.request.urlopen(req, timeout=connect_timeout) as resp:
                total = resp.length
                if total is None and resp.headers.get("Content-Length"):
                    total = int(resp.headers["Content-Length"])
                grand_total = (total + already) if total else None
                mode = "ab" if already > 0 else "wb"
                with open(part, mode) as f:
                    downloaded = already
                    last_log = time.time()
                    last_log_bytes = downloaded
                    while True:
                        chunk = resp.read(chunk_size)
                        if not chunk:
                            break
                        f.write(chunk)
                        downloaded += len(chunk)
                        now = time.time()
                        if now - last_log >= 5.0:
                            speed = (downloaded - last_log_bytes) / (now - last_log) / 1e6
                            if grand_total:
                                pct = 100.0 * downloaded / grand_total
                                print(f"     {downloaded/1e6:7.1f}/{grand_total/1e6:7.1f} MB ({pct:.0f}%) {speed:.1f} MB/s")
                            else:
                                print(f"     {downloaded/1e6:7.1f} MB {speed:.1f} MB/s")
                            last_log = now
                            last_log_bytes = downloaded
            os.replace(part, dest)
            print(f"   OK {dest.name} ({dest.stat().st_size/1e6:.0f} MB)")
            return True
        except urllib.error.HTTPError as e:
            if e.code in (503, 504, 429):
                wait = min(60, 2**attempt); print(f"   HTTP {e.code} retry {wait}s"); time.sleep(wait)
            elif e.code == 416:
                os.replace(part, dest); return True
            else:
                print(f"   HTTP {e.code}: {e.reason}"); return False
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            wait = min(60, 2**attempt); print(f"   reseau retry {wait}s ({type(e).__name__})"); time.sleep(wait)
        except Exception as e:
            print(f"   ERREUR {type(e).__name__}: {e}"); return False
    return False

# 6. Telechargements train (HR, LR ACCESS-CM2)
if not Path(HR_PATH).exists():
    print(f"Download HR ACCESS-CM2: {HR_PATH}")
    if not stream_download(URL_ZENODO_HR, HR_PATH):
        raise RuntimeError("Echec download HR ACCESS-CM2")
if not Path(LR_PATH).exists():
    print(f"Download LR ACCESS-CM2: {LR_PATH}")
    if not stream_download(URL_ZENODO_LR, LR_PATH):
        raise RuntimeError("Echec download LR ACCESS-CM2")

# 7. Telechargements test (EC-Earth3, NorESM2-MM)
TEST_ROOT = DATA_ROOT / "test"
TEST_ROOT.mkdir(parents=True, exist_ok=True)
for _filename, _url in URLS_TEST:
    _filepath = TEST_ROOT / _filename
    if _filepath.exists():
        continue
    print(f"Download test: {_filename}")
    if not stream_download(_url, str(_filepath)):
        raise RuntimeError(f"Echec download {_filename}")

# 8. Statics + normalization (fallback gdown si Drive public)
_PUBLIC_DRIVE_FALLBACKS = {
    "static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc":
        "1KY6IS1W5Wt-l_xyV7Qw8caA49zPzuSEx",
    "normalization_coefs/mean_1974_2011.nc":
        "14wVaJTUDgLwLlFcqRFA6pzJg9tZtAVQ0",
    "normalization_coefs/std_1974_2011.nc":
        "1ycqq9DqpfdOOiyQqgKs797OzRdHND3ZL",
}

def _gdown_install():
    try:
        import gdown; return True
    except ImportError:
        import subprocess
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"], timeout=120)
            import gdown; return True
        except Exception:
            return False

def _try_gdown(path):
    if not path:
        return False
    pth = Path(path)
    rel_key = None
    for _key in _PUBLIC_DRIVE_FALLBACKS:
        if str(pth).endswith(_key.replace("/", os.sep)) or str(pth).endswith(_key):
            rel_key = _key; break
    if rel_key is None:
        return False
    file_id = _PUBLIC_DRIVE_FALLBACKS[rel_key]
    pth.parent.mkdir(parents=True, exist_ok=True)
    if not _gdown_install():
        return False
    import gdown
    try:
        print(f"   gdown.download(id={file_id}) -> {pth}")
        gdown.download(id=file_id, output=str(pth), quiet=False)
        return pth.exists() and pth.stat().st_size > 0
    except Exception as e:
        print(f"   gdown ERREUR: {e}"); return False

for _var, _name in [("STATIC_PATH", "Static"), ("MEAN_PATH", "Mean"), ("STD_PATH", "Std")]:
    _p = globals()[_var]
    if _p and Path(_p).exists():
        continue
    if _try_gdown(_p):
        print(f"   OK {_name} (gdown public)")
    else:
        print(f"   {_name} absent: {_p} -> None")
        globals()[_var] = None

# 9. Resume final
print()
print("Datasets disponibles :")
print(f"  LR train  : {LR_PATH}  ({'OK' if Path(LR_PATH).exists() else 'MISSING'})")
print(f"  HR train  : {HR_PATH}  ({'OK' if Path(HR_PATH).exists() else 'MISSING'})")
print(f"  Static    : {STATIC_PATH}  ({'OK' if STATIC_PATH and Path(STATIC_PATH).exists() else 'NONE'})")
print(f"  Mean/Std  : {MEAN_PATH} / {STD_PATH}")
for fname, _ in URLS_TEST:
    p = TEST_ROOT / fname
    print(f"  Test      : {p.name}  ({'OK' if p.exists() else 'MISSING'})")

# 10. GCM_REGISTRY pour OOD
GCM_REGISTRY = {
    "ACCESS-CM2":  (Path(LR_PATH), Path(HR_PATH), True),
    "EC-Earth3":   (TEST_ROOT / "EC-Earth3_histupdated_compressed.nc",
                     TEST_ROOT / "EC-Earth3_historical_precip_compressed.nc", False),
    "NorESM2-MM":  (TEST_ROOT / "NorESM2-MM_histupdated_compressed.nc",
                     TEST_ROOT / "NorESM2-MM_historical_precip_compressed.nc", False),
}

# 11. Pipeline + builder
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder

def make_pipeline(lr_path, hr_path):
    return NetCDFDataPipeline(
        lr_path=str(lr_path), hr_path=str(hr_path),
        static_path=str(STATIC_PATH) if STATIC_PATH and Path(STATIC_PATH).exists() else None,
        seq_len=int(CONFIG.data.seq_len),
        baseline_strategy=str(CONFIG.data.baseline_strategy),
        baseline_factor=int(CONFIG.data.baseline_factor),
        target_transform=str(CONFIG.data.get("target_transform", "log1p")),
        normalize=bool(CONFIG.data.normalize),
        nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
        precipitation_delta=float(CONFIG.data.get("precipitation_delta", 0.01)),
        lr_variables=list(CONFIG.data.lr_variables),
        hr_variables=list(CONFIG.data.hr_variables),
        static_variables=list(CONFIG.data.static_variables) if STATIC_PATH and Path(STATIC_PATH).exists() else None,
        means_path=str(MEAN_PATH) if MEAN_PATH and Path(MEAN_PATH).exists() else None,
        stds_path=str(STD_PATH) if STD_PATH and Path(STD_PATH).exists() else None,
        eager_load_datasets=bool(CONFIG.data.get("eager_load_datasets", False)),
    )

pipeline_access = make_pipeline(LR_PATH, HR_PATH)
builder = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline_access.get_static_dataset(),
    include_mid_layer=bool(CONFIG.graph.include_mid_layer),
)
print()
print(f"[OK] Builder cree ({len(builder.dynamic_node_types)} dyn + {len(builder.static_node_types)} static)")

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    dynamic_features = {nt: lr_nodes_steps[0] for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {"lr": lr_tensor, "residual": sample["residual"],
            "baseline": sample.get("baseline"), "hetero": hetero}

test_dataset = pipeline_access.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len), stride=int(CONFIG.data.stride), as_torch=True,
)
sample = next(iter(test_dataset))
_runtime_dim = int(sample["lr"].shape[1])
if _runtime_dim != int(CONFIG.rcn.driver_dim):
    CONFIG.rcn.driver_dim = _runtime_dim
    CONFIG.rcn.reconstruction_dim = _runtime_dim
    print(f"[INFO] CONFIG.rcn.driver_dim -> {_runtime_dim}")

# 12. Stacks
from st_cdgm.models import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
    GraphToGridDecoder, RCNCell, RCNSequenceRunner,
    CausalDiffusionDecoder,
)
from st_cdgm.models.edm_preconditioner import EDMConfig
try:
    from st_cdgm.models import ConditionalSkipBlock
    SKIP_AVAILABLE = True
except ImportError:
    SKIP_AVAILABLE = False
    ConditionalSkipBlock = None

def build_stack(ckpt_path, name):
    print(f"  [{name}] {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    # 1. Encoder configs depuis CONFIG.encoder.metapaths (filtre selon allowed_nodes du builder)
    allowed_nodes = set(builder.dynamic_node_types + builder.static_node_types)
    encoder_configs = []
    for _mp in CONFIG.encoder.metapaths:
        _src, _rel, _tgt = _mp.src, _mp.relation, _mp.target
        if _src in allowed_nodes and _tgt in allowed_nodes:
            encoder_configs.append(IntelligibleVariableConfig(
                name=_mp.name,
                meta_path=(_src, _rel, _tgt),
                pool=_mp.get("pool", "mean"),
            ))
    # Ajoute le static metapath si pipeline a static_dataset
    if pipeline_access.get_static_dataset() is not None:
        encoder_configs.append(IntelligibleVariableConfig(
            name="static", meta_path=("SP_HR", "causes", "GP850"), pool="mean",
        ))

    enc = IntelligibleVariableEncoder(
        configs=encoder_configs,
        hidden_dim=CONFIG.encoder.hidden_dim,
        conditioning_dim=CONFIG.encoder.conditioning_dim,
    ).to(DEVICE)
    num_vars = len(encoder_configs)

    rcn_cell = RCNCell(
        num_vars=num_vars,
        hidden_dim=CONFIG.rcn.hidden_dim,
        driver_dim=int(CONFIG.rcn.driver_dim),
        reconstruction_dim=int(CONFIG.rcn.reconstruction_dim),
        dropout=CONFIG.rcn.dropout,
    ).to(DEVICE)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

    rh = GraphToGridDecoder(
        d_model=CONFIG.encoder.hidden_dim,
        hr_h=CONFIG.graph.hr_shape[0], hr_w=CONFIG.graph.hr_shape[1],
    ).to(DEVICE)

    edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
    # Convertit unet_kwargs en dict + tuples pour block_types
    from omegaconf import OmegaConf as _OC
    _unet_kwargs = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for _k in ("down_block_types", "up_block_types"):
        if _k in _unet_kwargs and isinstance(_unet_kwargs[_k], list):
            _unet_kwargs[_k] = tuple(_unet_kwargs[_k])

    hr_channels = int(sample["residual"].shape[1])

    diff = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height),
        width=int(CONFIG.diffusion.width),
        unet_kwargs=_unet_kwargs,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=bool(CONFIG.diffusion.get("use_gradient_checkpointing", False)),
        conv_padding_mode=str(CONFIG.diffusion.get("conv_padding_mode", "zeros")),
        anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
        edm_config=edm_cfg,
        causal_concat=True,
    ).to(DEVICE)
    def _safe_load(name, module):
        """Load state_dict avec checks None + dict-like + strip prefixes."""
        key = f"{name}_state_dict"
        if key not in ckpt:
            print(f"  [{name}] [WARN] {key} absent du checkpoint")
            return False
        sd = ckpt[key]
        if sd is None:
            print(f"  [{name}] [WARN] {key} is None - skip")
            return False
        if not hasattr(sd, "items"):
            print(f"  [{name}] [WARN] {key} not dict-like ({type(sd).__name__}) - skip")
            return False
        # Strip prefixes torch.compile ('_orig_mod.') et DDP ('module.')
        prefixes = ["_orig_mod.", "module."]
        stripped = {}
        for kk, vv in sd.items():
            new_k = kk
            for p in prefixes:
                if new_k.startswith(p):
                    new_k = new_k[len(p):]
            stripped[new_k] = vv
        try:
            missing, unexpected = module.load_state_dict(stripped, strict=False)
            if missing:
                print(f"  [{name}] [INFO] {len(missing)} keys manquantes (premieres : {missing[:3]})")
            if unexpected:
                print(f"  [{name}] [INFO] {len(unexpected)} keys inattendues (premieres : {unexpected[:3]})")
            return True
        except Exception as e:
            print(f"  [{name}] [ERREUR] load_state_dict: {type(e).__name__}: {e}")
            return False

    for n, m in [("encoder", enc), ("rcn_cell", rcn_cell),
                  ("regression_head", rh), ("diffusion", diff)]:
        _safe_load(n, m)
    skip = None
    if (SKIP_AVAILABLE and "skip_block_state_dict" in ckpt
            and ckpt["skip_block_state_dict"] is not None):
        skip = ConditionalSkipBlock(
            lr_channels=len(CONFIG.data.lr_variables),
            hr_shape=tuple(CONFIG.graph.hr_shape),
        ).to(DEVICE)
        try:
            skip.load_state_dict(ckpt["skip_block_state_dict"], strict=False)
            print(f"  [{name}] [+] skip_block ({skip.num_params()} params)")
        except Exception as e:
            print(f"  [{name}] [WARN] skip_block load failed: {e}")
            skip = None
    enc.eval(); rcn_cell.eval(); rh.eval(); diff.eval()
    if skip is not None:
        skip.eval()
    A_dag = rcn_cell.A_dag.detach().cpu().clone() if hasattr(rcn_cell, "A_dag") else None
    return {"encoder": enc, "rcn_runner": rcn_runner, "regression_head": rh,
            "diffusion": diff, "skip_block": skip, "A_dag": A_dag, "variant": name}

print()
print("Chargement des stacks...")
t0 = time.time()
# Phase F (2026-06-11) : ORACLE_DIR resolu via EVAL_VERSION (baseline / finetuned).
# Le baseline CorrDiff (NONCAUSAL_DIR) reste sur "epoch_last.pth" (jamais touche).
stack_v5 = build_stack(ORACLE_DIR / f"{CHECKPOINT_NAME}.pth", "Oracle")
stack_nc = build_stack(NONCAUSAL_DIR / "epoch_last.pth", "CorrDiff")
print(f"[OK] 2 stacks charges en {time.time()-t0:.1f}s")

# 13. Predict generique
@torch.no_grad()
def predict_with_stack(stack, batch, K=4, n_steps=32):
    enc, rcn, rh, diff, skip = (stack["encoder"], stack["rcn_runner"],
                                  stack["regression_head"], stack["diffusion"],
                                  stack["skip_block"])
    lr = batch["lr"].to(DEVICE)
    H_init = enc.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr[t] for t in range(lr.shape[0])]
    seq = rcn.run(H_init, drivers, reconstruction_sources=None)
    H_T = seq.states[-1]
    mu_c = rh(H_T)
    tshape = batch["residual"][-1].to(DEVICE).shape
    if tshape[-2:] != mu_c.shape[-2:]:
        mu_c = torch.nn.functional.interpolate(
            mu_c, size=tshape[-2:], mode="bilinear", align_corners=False,
        )
    if skip is not None:
        lr_last = drivers[-1] if drivers[-1].dim() == 4 else drivers[-1].unsqueeze(0)
        mu, _ = skip(lr_last, mu_c)
    else:
        mu = mu_c
    mu = torch.nan_to_num(mu, nan=0.0)
    bl = batch["baseline"][-1].to(DEVICE)
    if bl.dim() == mu.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)
    ens = []
    for _ in range(K):
        o = diff.sample(
            conditioning=None, num_steps=n_steps,
            scheduler_type="edm_karras", apply_constraints=False,
            mu_HR=mu, baseline_log=bl,
        )
        r = o.residual if hasattr(o, "residual") else o
        ens.append((bl + mu + r).cpu())
    return torch.stack(ens, dim=0)

print()
print("=" * 70)
print("Bootstrap autonome COMPLET")
print("=" * 70)
print("Variables disponibles :")
print(f"  CONFIG, DEVICE, builder, convert_sample_to_batch, predict_with_stack")
print(f"  stack_v5, stack_nc, GCM_REGISTRY, make_pipeline")
print(f"  test_dataset (ACCESS-CM2 in-dist)")
print()
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"TEST_ROOT  : {TEST_ROOT}")


---
## Helpers probabilistes (verbatim du notebook d'évaluation, Phase 7)
CRPS empirique, spread/skill, CRPS-SS, RMSE en mm/jour — identiques aux
métriques probabilistes du mémoire.

In [ ]:
# ============================================================
# Phase 7 : runs OOD reels avec run_aligned_eval (vendored Rampal)
# + sidecar probabilistic_metrics : CRPS, RMSE, spread, CRPS-SS, rank histogram
# Sortie 1 : aligned_metrics_<GCM>_<variant>.json   (indices climatiques)
# Sortie 2 : probabilistic_metrics_<GCM>_<variant>.json (ensemble-based)
# ============================================================
import json
import time
import torch
import numpy as np
from st_cdgm.evaluation.aligned_eval import run_aligned_eval

# Parametres
N_TIMES_OOD = 365
K_SAMPLES_OOD = 12          # bump 4 -> 12 pour CRPS empirique non bruite
N_STEPS_DIFF = 18

# --- Metriques probabilistes ---------------------------------------------

def _crps_empirical_fast(samples, obs):
    """CRPS empirique vectorise via tri (O(K log K) par point).

    samples : (K, ...) array, ensemble
    obs     : (...,) array, observation
    return  : (...,) array, CRPS par point
    Formule : E|X - y| - 0.5 * E|X - X'|, avec
              0.5 * (1/K^2) * sum_ij |xi - xj| = (1/K^2) * sum_k (2k - K - 1) * x_(k)
    """
    K = samples.shape[0]
    term1 = np.nanmean(np.abs(samples - obs[None]), axis=0)
    s = np.sort(samples, axis=0)
    k_idx = np.arange(1, K + 1).reshape((K,) + (1,) * (s.ndim - 1)).astype(np.float64)
    weights = 2.0 * k_idx - K - 1.0
    term2 = np.sum(weights * s, axis=0) / (K * K)
    return term1 - term2


def _crps_clim_per_pixel(truth):
    """CRPS de la climato empirique (distribution par pixel sur l'axe temps).

    Pour X, X' iid ~ distribution-truth(h,w) et y ~ idem :
        CRPS_clim(h,w) = E|X - y| - 0.5 * E|X - X'| = 0.5 * E|X - X'|
    (car E|X-Y| = E|X-X'| pour des copies iid).
    """
    T = truth.shape[0]
    t_sorted = np.sort(truth, axis=0)
    k_idx = np.arange(1, T + 1).reshape((T, 1, 1)).astype(np.float64)
    weights = 2.0 * k_idx - T - 1.0
    return np.sum(weights * t_sorted, axis=0) / (T * T)


def _rank_histogram(samples, truth):
    """Histogramme de Talagrand : rang de truth parmi les K samples (K+1 bins)."""
    K = samples.shape[0]
    rank = (samples < truth[None]).sum(axis=0).astype(np.int64)
    hist, _ = np.histogram(rank.flatten(), bins=np.arange(K + 2) - 0.5)
    return hist.astype(int).tolist()


def probabilistic_metrics(ens_log1p, truth_log1p):
    """Calcule toutes les metriques probabilistes apres conversion log1p -> mm/jour.

    ens_log1p   : (K, T, H, W) ensemble en espace log1p
    truth_log1p : (T, H, W) verite en espace log1p
    """
    ens = np.expm1(np.clip(ens_log1p.astype(np.float64), 0.0, None))
    truth = np.expm1(np.clip(truth_log1p.astype(np.float64), 0.0, None))

    pred_mean = ens.mean(axis=0)                         # (T, H, W)
    err2 = (pred_mean - truth) ** 2
    rmse_global = float(np.sqrt(np.nanmean(err2)))
    rmse_map_t = np.sqrt(np.nanmean(err2, axis=0))       # (H, W)

    ens_var = ens.var(axis=0)                            # (T, H, W)
    spread_global = float(np.sqrt(np.nanmean(ens_var)))
    spread_skill_ratio = float(spread_global / max(rmse_global, 1e-9))

    crps_model = _crps_empirical_fast(ens, truth)        # (T, H, W)
    crps_model_global = float(np.nanmean(crps_model))

    crps_clim_map = _crps_clim_per_pixel(truth)          # (H, W)
    crps_clim_global = float(np.nanmean(crps_clim_map))

    crps_ss = 1.0 - crps_model_global / max(crps_clim_global, 1e-9)

    hist = _rank_histogram(ens, truth)
    K = int(ens.shape[0])
    expected_per_bin = float(truth.size / (K + 1))
    chi2_uniform = float(sum((c - expected_per_bin) ** 2 / expected_per_bin for c in hist))

    return {
        "K_samples": K,
        "n_times": int(ens.shape[1]),
        "grid": [int(truth.shape[-2]), int(truth.shape[-1])],
        "rmse_global_mm": rmse_global,
        "rmse_map_mean_mm": float(np.nanmean(rmse_map_t)),
        "rmse_map_max_mm": float(np.nanmax(rmse_map_t)),
        "spread_global_mm": spread_global,
        "spread_skill_ratio": spread_skill_ratio,
        "crps_model_global_mm": crps_model_global,
        "crps_clim_global_mm": crps_clim_global,
        "crps_skill_score": float(crps_ss),
        "rank_histogram": hist,
        "rank_histogram_bins": list(range(len(hist))),
        "rank_histogram_chi2_vs_uniform": chi2_uniform,
        "_caveat": (
            "CRPS_clim computed from test-truth empirical distribution per pixel "
            "(includes the day under evaluation; slight optimistic bias for T~365). "
            "spread_skill_ratio ~1 = well-calibrated, <1 = under-dispersive, >1 = over-dispersive."
        ),
    }


---
## Évaluation par tranche climatique

Chaque tranche passe par un pipeline **borné par dates explicites**, par
slicing xarray en place des quatre datasets internes (`lr_dataset`,
`baseline_prepared`, `residual_dataset`, `hr_dataset`) que
`build_sequence_dataset` lit au moment de l'appel — méthode compatible avec la
branche `two-stage-causal` (V5), qui ne connaît pas le mécanisme K9
(`split=...`). La normalisation LR (statistiques d'entraînement 1974--2011)
est appliquée AVANT le slice : aucun réajustement sur les tranches décalées.

Garde-fous : chaque slice imprime ses bornes réelles et deux asserts vérifient
qu'il est non vide et strictement inclus dans les dates demandées. Les
tranches `id` et `cold` prennent les 120 premiers jours de leur période
(mêmes mois calendaires : saisonnalité appariée). La queue chaude est
sélectionnée par le z-score t850 moyen domaine au dernier pas de chaque
séquence, et l'ordre climatique final `cold < id < warm` est vérifié par
assert avant toute évaluation.

In [ ]:
# ============================================================
# OOD directionnel : cold-shift reel + warm-tail proxy
# ============================================================
import json, time
import numpy as np
import torch
import matplotlib.pyplot as plt

N_DAYS   = 120   # jours evalues par tranche   (90 sur L4)
K_ENS    = 8     # membres d'ensemble          (6 sur L4)
N_STEPS  = 14    # pas de diffusion
WARM_FRAC = 0.07 # fraction des jours les plus chauds (pool 2012-2014)

LR_VARS = list(CONFIG.data.lr_variables)
T850_IDX = LR_VARS.index("t_850")

def make_pipeline_dated(start, end):
    """Pipeline ACCESS-CM2 borne a [start, end].

    Compatible branche two-stage-causal (V5) : son pipeline n'a pas le
    mecanisme K9 (split=...), on slice donc EN PLACE les datasets xarray que
    build_sequence_dataset lit au moment de l'appel. La normalisation LR est
    deja appliquee (stats train externes) avant le slice : aucun reajustement.
    """
    lr_path, hr_path, _ = GCM_REGISTRY["ACCESS-CM2"]
    pipe = make_pipeline(lr_path, hr_path)   # helper du bootstrap (Cell 4)
    tdim = pipe.dims.time
    n_before = int(pipe.lr_dataset.sizes[tdim])
    for attr in ("lr_dataset", "baseline_prepared", "residual_dataset", "hr_dataset"):
        ds_a = getattr(pipe, attr, None)
        if ds_a is not None and tdim in ds_a.dims:
            setattr(pipe, attr, ds_a.sel({tdim: slice(start, end)}))
    n_after = int(pipe.lr_dataset.sizes[tdim])
    # Garde-fous anti-bug-silencieux : le slice doit etre non vide ET strict.
    assert 0 < n_after < n_before, (
        f"slice [{start}, {end}] douteux : {n_after}/{n_before} pas de temps "
        f"(0 = dates hors axe ; = n_before = slicing sans effet)")
    t0 = str(pipe.lr_dataset[tdim].values[0])[:10]
    t1 = str(pipe.lr_dataset[tdim].values[-1])[:10]
    print(f"  [slice] {start}..{end} -> {n_after} jours ({t0} .. {t1})")
    assert start[:10] <= t0 and t1 <= end[:10], (
        f"bornes non respectees : [{t0}, {t1}] hors [{start}, {end}]")
    return pipe

def collect_samples(start, end, max_days=None):
    """Charge les echantillons de la tranche en RAM (LR + residu + baseline)."""
    pipe = make_pipeline_dated(start, end)
    ds = pipe.build_sequence_dataset(seq_len=int(CONFIG.data.seq_len), stride=1, as_torch=True)
    out = []
    for s in ds:
        out.append(s)
        if max_days is not None and len(out) >= max_days:
            break
    if not out:
        raise RuntimeError(f"Tranche [{start}, {end}] : aucun echantillon construit.")
    return out

def t850_score(sample):
    """z-score t850 moyen domaine au dernier pas de la sequence (stats train)."""
    last = sample["lr"][-1]               # [C, H, W] (grille) ou [N, C] (graphe)
    if last.dim() == 2:
        return float(last[:, T850_IDX].mean())
    return float(last[T850_IDX].mean())

@torch.no_grad()
def eval_days(stack, samples, label):
    ens_log, truth_log = [], []
    t0 = time.time()
    for i, s in enumerate(samples):
        b = convert_sample_to_batch(s, builder, DEVICE)
        ens = predict_with_stack(stack, b, K=K_ENS, n_steps=N_STEPS)
        while ens.dim() > 3:
            ens = ens.squeeze(1)
        ens_log.append(ens.cpu().numpy())
        truth_log.append((b["baseline"][-1] + b["residual"][-1]).cpu().squeeze().numpy())
        if (i + 1) % 40 == 0:
            print(f"    [{label}] {i+1}/{len(samples)} ({time.time()-t0:.0f}s)")
    ens_full = np.stack(ens_log, axis=1)
    truths = np.stack(truth_log, axis=0)
    return probabilistic_metrics(ens_full, truths)

# --- Construction des trois tranches --------------------------------------
print("Chargement des tranches (I/O uniquement, pas de GPU)...")
tranches = {}
tranches["id_2012_2013"]   = collect_samples("2012-01-01", "2013-12-31", max_days=N_DAYS)
tranches["cold_1960_1965"] = collect_samples("1960-01-01", "1965-12-31", max_days=N_DAYS)

# Queue chaude : pool 2012-2014 = test + holdout, hors train ET hors val
warm_pool = collect_samples("2012-01-01", "2014-12-31", max_days=None)
scores = np.array([t850_score(s) for s in warm_pool])
n_warm = min(N_DAYS, max(20, int(len(warm_pool) * WARM_FRAC)))
idx_warm = np.argsort(-scores)[:n_warm]
tranches["warm_tail_2012_2014"] = [warm_pool[i] for i in sorted(idx_warm)]
thr = float(np.sort(scores)[-n_warm])
print(f"warm tail : {n_warm} jours / {len(warm_pool)} (z-t850 >= {thr:+.2f}, "
      f"quantile {1 - n_warm/len(warm_pool):.2%})")
del warm_pool

# --- Sanity check climatique : les tranches doivent s'ordonner cold<id<warm --
zmeans = {k: float(np.mean([t850_score(s) for s in v])) for k, v in tranches.items()}
for k, v in tranches.items():
    print(f"  {k:22s}: {len(v):4d} jours | z-t850 moyen = {zmeans[k]:+.3f}")
assert zmeans["cold_1960_1965"] < zmeans["id_2012_2013"] < zmeans["warm_tail_2012_2014"], (
    f"Ordre climatique inattendu {zmeans} : le decoupage par tranches est suspect "
    "(bug de slicing ou de selection) -- NE PAS interpreter les resultats.")
print("[OK] ordre climatique cold < id < warm verifie")

# --- Evaluation 3 tranches x 2 modeles -------------------------------------
results = {}
t_glob = time.time()
for tname, samples in tranches.items():
    for mname, stack in [("Oracle", stack_v5), ("Noncausal", stack_nc)]:
        label = f"{mname}/{tname}"
        print(f"\n=== {label} ({len(samples)} jours) ===")
        m = eval_days(stack, samples, label)
        results[label] = {k: m[k] for k in
            ("crps_model_global_mm", "crps_skill_score", "spread_skill_ratio",
             "rmse_global_mm", "K_samples", "n_times")}
        print(f"    CRPS={m['crps_model_global_mm']:.4f}  spread/skill={m['spread_skill_ratio']:.3f}  "
              f"RMSE={m['rmse_global_mm']:.3f}")

# --- Synthese : niveaux + pente de degradation -----------------------------
print("\n" + "=" * 86)
print(f"{'Tranche':<24}{'CRPS Oracle':>13}{'CRPS Noncausal':>16}{'gain':>9}{'sp/sk O':>9}{'sp/sk N':>9}")
print("-" * 86)
summary = {}
for tname in tranches:
    o = results[f"Oracle/{tname}"]; n = results[f"Noncausal/{tname}"]
    gain = (n["crps_model_global_mm"] - o["crps_model_global_mm"]) / n["crps_model_global_mm"]
    summary[tname] = {"gain_crps_rel": gain}
    print(f"{tname:<24}{o['crps_model_global_mm']:>13.4f}{n['crps_model_global_mm']:>16.4f}"
          f"{gain:>8.1%}{o['spread_skill_ratio']:>9.3f}{n['spread_skill_ratio']:>9.3f}")
print("-" * 86)
oid = results["Oracle/id_2012_2013"]["crps_model_global_mm"]
nid = results["Noncausal/id_2012_2013"]["crps_model_global_mm"]
for tname in ("cold_1960_1965", "warm_tail_2012_2014"):
    do = results[f"Oracle/{tname}"]["crps_model_global_mm"] / oid - 1
    dn = results[f"Noncausal/{tname}"]["crps_model_global_mm"] / nid - 1
    summary[tname].update({"degradation_oracle": do, "degradation_noncausal": dn})
    print(f"pente {tname:<22}: Oracle {do:+.1%} vs Noncausal {dn:+.1%} "
          f"({'Oracle degrade MOINS' if do < dn else 'Oracle degrade PLUS'})")

out = RESULTS_DIR / "ood_climate_shift.json"
out.write_text(json.dumps({
    "protocol": {"n_days": N_DAYS, "K": K_ENS, "n_steps": N_STEPS,
                 "warm_pool": "2012-01-01..2014-12-31 (test+holdout, hors train/val)",
                 "warm_frac": WARM_FRAC, "warm_threshold_z": thr,
                 "z_t850_means": zmeans,
                 "normalisation": "stats train 1974-2011, aucun reajustement",
                 "caveat": "proxys directionnels (froid reel / queue chaude), pas de scenario SSP"},
    "checkpoints": CKPT_INFO,
    "results": results, "summary": summary,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[OK] {out}")

# --- Figure -----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
names = list(tranches.keys())
x = np.arange(len(names)); w = 0.36
co = [results[f"Oracle/{t}"]["crps_model_global_mm"] for t in names]
cn = [results[f"Noncausal/{t}"]["crps_model_global_mm"] for t in names]
axes[0].bar(x - w/2, co, w, label="Oracle", color="steelblue")
axes[0].bar(x + w/2, cn, w, label="Noncausal", color="orange")
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=12, fontsize=9)
axes[0].set_ylabel("CRPS (mm/j)"); axes[0].set_title("CRPS par tranche climatique")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
so = [results[f"Oracle/{t}"]["spread_skill_ratio"] for t in names]
sn = [results[f"Noncausal/{t}"]["spread_skill_ratio"] for t in names]
axes[1].bar(x - w/2, so, w, label="Oracle", color="steelblue")
axes[1].bar(x + w/2, sn, w, label="Noncausal", color="orange")
axes[1].axhline(1.0, color="red", ls="--", lw=1, label="calibration parfaite")
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=12, fontsize=9)
axes[1].set_ylabel("spread / RMSE"); axes[1].set_title("Calibration par tranche")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
figp = RESULTS_DIR / "ood_climate_shift.png"
plt.savefig(figp, dpi=140, bbox_inches="tight")
plt.close()
print(f"[OK] {figp}")
print(f"\nTermine en {(time.time()-t_glob)/60:.1f} min")
